# NEV Pricing Analysis Project — Post-Mortem

## 1. Project Overview

- **Objective**: Build a model to predict vehicle price (10k RMB) from configuration parameters (dimensions, battery, brand, ADAS level, etc.), and quantify marginal premiums of individual features to support pricing decisions. Expected metrics: regression R² ≥ 0.7, test MAE ≤ 30k RMB.
- **Findings**: In the budget segment (<150k RMB), consumers pay for configurations — longer wheelbase, larger battery directly reflect in price. In the premium segment (>350k RMB), consumers pay for brand, design, service, and scarcity.
  - The model fails in the high-price range (>350k) for two reasons: (1) extremely sparse samples; (2) features cover only physical parameters, missing soft premium factors (brand, design, marketing).
- **Actual results**:
  - Regression: Test R² < 0.5, MAE ≈ 80k RMB.
  - Classification (price brackets): overall accuracy ≈ 63%, precision for premium cars (>350k) ≈ 0.3.

- **Resources**:
  - Data: 265 manually collected records → 243 after cleaning.
  - Features: 18 raw → 30+ engineered, final model used 5 core features.
  - Iterations: Random Forest regression + grid search; later switched to Random Forest classification.
  - Total: ~2 weeks.

- **AI collaboration**:
  - Skepticism existed at the start about whether ~200 samples could produce a commercially viable pricing model. AI was optimistic, claiming Random Forest is robust to small samples and 200 would suffice for premium analysis.
  - That skepticism was not independently verified; the decision to proceed was influenced by AI's confidence.
  - Throughout execution, all major decisions — cleaning strategy, feature engineering order, method selection (missing value handling, brand grouping, target encoding, feature selection) — were made independently.
  - AI served as a code generator under instruction, never proactively flagging small-sample risks.
  - **Key takeaway**: AI is an efficient coding assistant but cannot replace independent judgment. Over-reliance on AI's optimism was a root cause of the project's failure.

**Conclusion**: The project failed to meet its original business objectives, but the process offers high retrospective value.

---

## 2. Root Cause Analysis

### Data Level (Primary Cause)

- **Insufficient sample size**: 243 rows, 80+ brands — fewer than 3 cars per brand on average. Cannot capture brand premium or configuration-level effects. Pairwise comparisons (same brand, same segment, single config difference) are impossible.
- **Highly skewed target**: Price skewness = 3.9, kurtosis = 21.4. Only 5 cars >500k RMB; the model cannot learn high-price patterns.
- **Collinearity between price and size**: Wheelbase, length, width are strongly correlated with vehicle segment, which directly determines price. The model essentially learns "bigger car → higher price", not genuine configuration premiums.

### Feature Level

- **Target encoding for brand fails under small samples**: Using target encoding on 80+ brands (each with very few samples) produces unstable mean estimates. Though not true data leakage (only training means are used), the model over-relies on these noisy encoded values and ignores real brand value signals → severe overfitting.
- **Selected features still too weak**: Even after keeping only 5 important features (wheelbase, battery capacity, width, length, encoded brand category), the remaining predictors are still size proxies. Non-size factors (brand image, marketing, design) are completely lost.

### Model Level

- **Random Forest cannot overcome the small-sample ceiling**: After feature selection, 200 samples are still insufficient for learning stable non-linear patterns.

### Evaluation Level

- **Misaligned metric**: R² is dominated by outliers and squares large errors on expensive cars. A more relevant metric would be segment-wise MAE (e.g., error in the 100-200k bracket).

---

## 3. What Was Done

Throughout the project, data cleaning, feature engineering, and model diagnostics were guided by independent analysis rather than AI suggestions.

### 3.1 Data Cleaning — Tailored Missing Value Handling

**a) Physical-relationship-based imputation for wheelbase**  
`wheel_base` had few missing values. Deleting them would shrink an already tiny dataset; median imputation would erase real differences. A strong linear correlation (>0.7) between `wheel_base` and `length_mm` was observed. A linear regression using `length_mm` was built to predict missing `wheel_base`, preserving genuine dimensional relationships.

**b) Differentiated strategies for different columns**  
- For `battery_capacity_avg` and `range_CLTC_avg`: median by `segment` — cars in the same segment naturally have similar battery and range.
- For `battery_type` (categorical): filled with "Unknown", keeping missing information as a separate category.
- For `model`: removed rows (only 2) because it has too many unique values and weak correlation with price.

Outcome: maximised usable data without injecting false signals.

### 3.2 Feature Engineering

**a) Target handling**  
- Log transformation on `price_avg` (skewness 3.9 → near normal) to stabilise variance.
- After the initial split, a 60k RMB difference in mean price between train and test sets was discovered. `StratifiedShuffleSplit` (based on price bins) was introduced to force similar distributions — a step done before final modelling.

**b) Encoding strategy aligned with model choice**  
- Random Forest was chosen because it is insensitive to collinearity, does not assume normality, and captures interactions automatically — suitable for small-sample mixed data.
- Low-cardinality categoricals (`powertrain`, `ADAS_Level`, etc.) → one-hot.
- High-cardinality `brand_group` → target encoding (a reasonable attempt at the time, though it failed under small samples).
- Normality tests, homoscedasticity checks, VIF were consciously skipped — unnecessary for tree-based models.

**c) Feature selection**  
A full-feature Random Forest was trained first, then features with importance >0.01 were extracted. Final set: wheelbase, battery capacity, width, length, encoded brand category.

### 3.3 Active Diagnosis & Course Correction

- When test R² turned out very low, the first step was not hyperparameter tuning but checking train/test price distributions — informed by earlier EDA findings. After identifying the large discrepancy, stratified sampling was applied.
- In the classification variant, price brackets almost perfectly overlap with `segment` — meaning the model was learning "size = price", not genuine pricing logic. This insight stopped useless tuning and redirected to qualitative conclusions.

### 3.4 Domain-Driven Feature Extraction

Manual rules were designed to extract structured features from unstructured text:
- `dimensions` string → length, width, height, wheelbase.
- Seat count from `body_structure`.
- `radar_type` → binary `has_lidar`.
- `battery_type` → NCM / LFP / BOTH / Other.

These rules ensured feature realism and usability.

---

## 4. When Can Small-Sample ML Succeed?

### 4.1 Inherently Friendly Tasks

| Type | Key characteristic | Example | Why this project differs |
|------|--------------------|---------|--------------------------|
| Clear physical law | Deterministic or low-noise relation | Vibration pattern → bearing fault | Car pricing is heavily influenced by social/marketing factors — high noise. |
| Strong pre-trained prior | Task similar to large-scale public task | Medical image diagnosis (ImageNet); NLP (BERT) | No large-scale pre-trained model for "vehicle price from specs". |
| Explicit rule | Simple formula can express the relation | Material property (composition → strength) | Price is not a simple function of specs; includes consumer psychology and market competition. |

**Conclusion**: This task belongs to none of these friendly categories — failure was highly likely from the start.

### 4.2 Technical Remedies

| Method | Principle | Prerequisite | Why not applicable |
|--------|-----------|--------------|--------------------|
| Bayesian methods | Prior distributions constrain parameters | Reasonable prior (e.g., plausible brand premium range) | No reliable prior; high computational cost. |
| Regularisation / simpler models | Reduce model capacity | Low-dimensional features | RF already has some regularisation, but still overfits. |
| Semi-supervised learning | Use unlabelled data | Easily obtainable unlabelled samples | Could not get large unlabelled car config data. |
| Data augmentation | Generate synthetic samples | Invariance under transformations | Car specs are not invariant (adding 10mm to wheelbase does not create a real new car). |
| Transfer learning | Reuse weights from related task | Existing relevant pre-trained model | No public pre-trained model for car pricing. |
| Meta-learning | Learn to learn from many small tasks | Many similar small tasks available | No multi-task environment available. |

**Takeaway**: For a high-noise, socially-driven problem without a pre-trained backbone, even advanced methods need far more data (1000+ samples) to be reliable.

---

## 5. Data Requirements for Success

To reach industrial usefulness (R² ≥ 0.8, segment-wise MAE ≤ 20k RMB), future data must satisfy:

### 5.1 Sample Size & Coverage
- **Total**: ≥1000+ records.
- **Per brand**: ≥20-50 records per major brand, covering different segments (A-class, B-class, SUV, etc.).
- **Price segments**: each bracket (<150k, 150-250k, 250-400k, >400k) ≥15% of samples.

### 5.2 Feature Structure
- **Paired comparisons** for configuration premiums: same brand, same segment, similar model year, only one differing feature (e.g., sunroof vs. no sunroof). This is the gold standard.
- **Independent brand value metrics**: brand awareness score, average discount rate, dealer network density.
- **Real transaction signals**: actual transaction price (not MSRP), dealer discount, inventory depth, regional sales volume.
- **Time dimension**: ≥3-5 years of repeated observations to capture facelifts, depreciation, and competitive responses.

---

## 6. Key Lessons & Improvements

### Short-Term (if restarting)
- Scope was too broad — trying to cover 50k to 2M RMB with a single model. If restarted, focus on 0-500k market (mainstream), exclude luxury cars where pricing logic fundamentally differs and sample is too sparse. Narrower scope yields higher reliability and practical business value.

### Long-Term Capability Building
- **Feasibility gate** before starting: evaluate whether sample size supports the required analysis.
- **Mandatory EDA**: always check target skewness, class frequencies, train/test distribution homogeneity.
- **Failure pre-mortem**: define termination conditions up front (e.g., "if test R² < 0.5 after baseline, stop tuning").
- **AI collaboration rules**: for any advanced method suggested by AI (e.g., target encoding), demand explanation of its assumptions and small-sample risks.

---

## 7. Next Actions

- **Restart?** No — this dataset cannot support the original business goal.
- **Archive**:
  - Cleaned dataset (with `price_log`, `price_category`) and feature engineering code.
  - Final classification model (6-class accuracy 63%) as a small-sample pipeline demonstration.
  - This post-mortem report in the portfolio as a complete failure case study.
